# Run Visibility Notebook

Open this notebook in VS Code over SSH. For any matplotlib output, use **Open in Plot Viewer** to get zoom and pan controls without copying files off the cluster.

Set `DATA_DIR` and optional `RUN_ID` in the next cell. Leave `RUN_ID = None` to load the latest run for that dataset.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import Markdown, display

MASK_VALUE = -2.0
KIN_DIM = 13

DATA_DIR = "syndata"
RUN_ID = None  # None -> latest run under data/outputs/<DATA_DIR>
EVAL_MODE = None  # None -> infer from run_summary.txt when possible


def find_project_root(start=None):
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root containing pyproject.toml")


def resolve_run_id(outputs_root, data_dir, run_id=None):
    run_root = outputs_root / data_dir
    if not run_root.exists():
        raise FileNotFoundError(f"No outputs found for data_dir={data_dir!r}: {run_root}")
    if run_id is not None:
        run_dir = run_root / run_id
        if not run_dir.exists():
            raise FileNotFoundError(f"Run {run_id!r} not found under {run_root}")
        return run_id
    runs = sorted(path.name for path in run_root.iterdir() if path.is_dir())
    if not runs:
        raise FileNotFoundError(f"No run directories found under {run_root}")
    return runs[-1]


def infer_eval_mode(summary_text):
    label_map = {
        "clean reconstruction input": "clean",
        "masked kinematics input": "masked-kinematics",
        "masked imu_t input": "masked-imu-t",
    }
    for line in summary_text.splitlines():
        if not line.startswith("Evaluation input:"):
            continue
        for label, mode in label_map.items():
            if label in line:
                return mode
    return None


def mask_imu_t_features(data, kin_dim, imu_dim, mask_value=MASK_VALUE):
    masked = np.array(data, copy=True)
    imu_t_end = kin_dim + (imu_dim // 2)
    masked[:, kin_dim:imu_t_end] = mask_value
    return masked


def mask_kinematics_features(data, kin_dim, mask_value=MASK_VALUE):
    masked = np.array(data, copy=True)
    masked[:, :kin_dim] = mask_value
    return masked


def feature_label(index, kin_dim, imu_dim):
    if index < kin_dim:
        return f"feature {index:02d} | kinematics[{index}]"
    local_index = index - kin_dim
    imu_half = imu_dim // 2
    if local_index < imu_half:
        return f"feature {index:02d} | imu_t[{local_index}]"
    return f"feature {index:02d} | imu_tminus1[{local_index - imu_half}]"


PROJECT_ROOT = find_project_root()
OUTPUTS_ROOT = PROJECT_ROOT / "data" / "outputs"
ML_ROOT = PROJECT_ROOT / "data" / "ml"

resolved_run_id = resolve_run_id(OUTPUTS_ROOT, DATA_DIR, RUN_ID)
run_dir = OUTPUTS_ROOT / DATA_DIR / resolved_run_id
results_dir = run_dir / "results"
plots_dir = run_dir / "plots"
formatted_data_dir = ML_ROOT / DATA_DIR

summary_path = results_dir / "run_summary.txt"
summary_text = summary_path.read_text() if summary_path.exists() else ""
effective_eval_mode = EVAL_MODE or infer_eval_mode(summary_text) or "masked-kinematics"

test_data = np.load(formatted_data_dir / "test_data.npy")
output_path = results_dir / "output_data.csv"
if not output_path.exists():
    raise FileNotFoundError(f"Missing output_data.csv for run {resolved_run_id}: {output_path}")
output_data = np.loadtxt(output_path, delimiter=",")
if output_data.ndim == 1:
    output_data = output_data.reshape(1, -1)

training_curves_path = results_dir / "training_curves.csv"
if training_curves_path.exists():
    training_curves = pd.read_csv(training_curves_path)
else:
    training_curves = pd.DataFrame(columns=["epoch", "cost", "recon", "latent"])

input_dim = test_data.shape[1]
imu_dim = input_dim - KIN_DIM

if effective_eval_mode == "clean":
    eval_input = test_data
    eval_label = "clean reconstruction input"
elif effective_eval_mode == "masked-imu-t":
    eval_input = mask_imu_t_features(test_data, KIN_DIM, imu_dim)
    eval_label = "masked imu_t input"
else:
    eval_input = mask_kinematics_features(test_data, KIN_DIM)
    eval_label = "masked kinematics input"

imu_t_end = KIN_DIM + (imu_dim // 2)
sqerr_total = (test_data - output_data) ** 2
sqerr_kin = (test_data[:, :KIN_DIM] - output_data[:, :KIN_DIM]) ** 2
sqerr_imu = (test_data[:, KIN_DIM:] - output_data[:, KIN_DIM:]) ** 2
sqerr_imu_t = (test_data[:, KIN_DIM:imu_t_end] - output_data[:, KIN_DIM:imu_t_end]) ** 2
sqerr_imu_tminus1 = (test_data[:, imu_t_end:] - output_data[:, imu_t_end:]) ** 2

generated_summary_lines = [
    "Run Summary",
    "",
    f"Dataset: {DATA_DIR}",
    f"Run ID: {resolved_run_id}",
    f"Train shape: unavailable in notebook fallback",
    f"Test shape: {test_data.shape}",
    f"Evaluation input: {eval_label}, clean target",
    "",
    f"MSE total: {float(np.mean(sqerr_total)):.8f}",
    f"MSE kinematics: {float(np.mean(sqerr_kin)):.8f}",
    f"MSE IMU: {float(np.mean(sqerr_imu)):.8f}",
    f"MSE IMU(t): {float(np.mean(sqerr_imu_t)):.8f}",
    f"MSE IMU(t-1): {float(np.mean(sqerr_imu_tminus1)):.8f}",
    f"STD total: {float(np.std(sqerr_total)):.8f}",
    f"STD kinematics: {float(np.std(sqerr_kin)):.8f}",
    f"STD IMU: {float(np.std(sqerr_imu)):.8f}",
    f"STD IMU(t): {float(np.std(sqerr_imu_t)):.8f}",
    f"STD IMU(t-1): {float(np.std(sqerr_imu_tminus1)):.8f}",
    "",
    f"Results CSV: {output_path}",
    f"Plots dir: {plots_dir}",
]
if not summary_text:
    summary_text = "\n".join(generated_summary_lines)

display(
    Markdown(
        f"""
### Loaded Run

- Project root: `{PROJECT_ROOT}`
- Dataset: `{DATA_DIR}`
- Run ID: `{resolved_run_id}`
- Eval mode: `{effective_eval_mode}`
- Test data shape: `{test_data.shape}`
- Output data shape: `{output_data.shape}`
- Results dir: `{results_dir}`
- Plots dir: `{plots_dir}`
        """
    )
)


In [ ]:
display(Markdown("## Run Summary"))
if summary_text:
    display(Markdown(f"```\n{summary_text}\n```"))
else:
    print("No run_summary.txt found for this run.")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
if training_curves.empty:
    ax.text(0.5, 0.5, "No training curve points were recorded.", ha="center", va="center")
    ax.set_axis_off()
else:
    ax.plot(training_curves["epoch"], training_curves["cost"], label="cost")
    ax.plot(training_curves["epoch"], training_curves["recon"], label="recon")
    ax.plot(training_curves["epoch"], training_curves["latent"], label="latent")
    ax.set_title("Training Curves")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.grid(True, alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
feature_options = [
    (feature_label(index, KIN_DIM, imu_dim), index)
    for index in range(test_data.shape[1])
]

feature_dropdown = widgets.Dropdown(
    options=feature_options,
    value=0,
    description="Feature",
    layout=widgets.Layout(width="450px"),
)
sample_window = widgets.IntRangeSlider(
    value=(0, min(1000, test_data.shape[0] - 1)),
    min=0,
    max=test_data.shape[0] - 1,
    step=1,
    description="Samples",
    continuous_update=False,
    layout=widgets.Layout(width="95%"),
)
stride_dropdown = widgets.Dropdown(
    options=[1, 2, 5, 10, 20, 50, 100],
    value=1,
    description="Stride",
)
show_input = widgets.Checkbox(value=True, description="show input")
show_original = widgets.Checkbox(value=True, description="show original")
show_reconstructed = widgets.Checkbox(value=True, description="show recon")
show_sqerr = widgets.Checkbox(value=True, description="show squared error")


def render_feature(feature, sample_window, stride, show_input, show_original, show_reconstructed, show_sqerr):
    start, stop = sample_window
    stop = max(stop, start)
    indices = np.arange(start, stop + 1, stride)
    if indices.size == 0:
        indices = np.array([start])

    nrows = 2 if show_sqerr else 1
    fig, axes = plt.subplots(nrows, 1, figsize=(12, 7 if show_sqerr else 4), sharex=True)
    if nrows == 1:
        axes = [axes]

    ax = axes[0]
    if show_reconstructed:
        ax.plot(indices, output_data[indices, feature], label="reconstructed")
    if show_original:
        ax.plot(indices, test_data[indices, feature], label="original")
    if show_input:
        ax.plot(indices, eval_input[indices, feature], label="evaluation input", linestyle="--", alpha=0.7)

    ax.set_title(feature_label(feature, KIN_DIM, imu_dim))
    ax.set_ylabel("Feature Value")
    ax.grid(True, alpha=0.3)
    if show_input or show_original or show_reconstructed:
        ax.legend(loc="best")

    if show_sqerr:
        sqerr = (test_data[indices, feature] - output_data[indices, feature]) ** 2
        axes[1].plot(indices, sqerr, color="tab:red", label="squared error")
        axes[1].set_xlabel("Sample Index")
        axes[1].set_ylabel("Squared Error")
        axes[1].grid(True, alpha=0.3)
        axes[1].legend(loc="best")
    else:
        ax.set_xlabel("Sample Index")

    plt.tight_layout()
    plt.show()


controls = widgets.VBox(
    [
        feature_dropdown,
        sample_window,
        widgets.HBox([stride_dropdown, show_input, show_original, show_reconstructed, show_sqerr]),
    ]
)
viewer = widgets.interactive_output(
    render_feature,
    {
        "feature": feature_dropdown,
        "sample_window": sample_window,
        "stride": stride_dropdown,
        "show_input": show_input,
        "show_original": show_original,
        "show_reconstructed": show_reconstructed,
        "show_sqerr": show_sqerr,
    },
)
display(Markdown("## Feature Explorer"))
display(controls, viewer)
